# K21 Academy — Feature Engineering with Python & AWS
### A Beginner-Friendly, Code-Based Hands-On Lab — **v2 (Fixed Edition)**

**Dataset:** K21 E-commerce Transactions (10,000 records)  
**Stack:** Python 3.10+ • pandas • scikit-learn • boto3 • SageMaker SDK  
**AWS Services:** S3 • SageMaker Notebook • Feature Store • Athena  
**Estimated Time:** 3 – 4 hours

---

### 🆕 What's new in v2

This version fixes every issue we hit during the first run:

- ✅ Graceful handling of S3 encryption permission errors
- ✅ Uses `sagemaker.default_bucket()` — works with any SageMaker role
- ✅ Robust NaN cleanup (no more `AssertionError: Still have NaNs!`)
- ✅ `CAT_HIGH` defined correctly from the start (no more `returnreason` KeyError)
- ✅ Feature Group **reconnect-by-name** pattern (survives kernel restarts)
- ✅ Ingestion with clear progress messages (no more "cell seems frozen")
- ✅ Cleanup cell that finds and deletes **all** leftover `k21-featureeng-*` buckets

---

### 📘 How to use this notebook

1. **Run cells one at a time** using `Shift + Enter`. Do not "Run All" — each cell builds on the previous one.
2. **Read the markdown notes** above every code block. They explain *what* the code does and *why*.
3. **Watch for ✅ CHECKPOINT cells** — these tell you what a successful output looks like.
4. If a cell fails, scroll up and make sure every earlier cell ran without a red error.

> 💡 **Tip:** This notebook runs on **SageMaker Studio** (`Python 3` kernel) or a classic **Notebook Instance** (`conda_python3` kernel). Both work.

> ⚠️ **IF YOUR KERNEL RESTARTS:** Just click **Kernel → Restart & Run All** and walk away for 5 min. Everything will re-run automatically.

---


## 🎯 What You Will Learn

By the end of this notebook you will have:

- ✅ Loaded a CSV from **Amazon S3** using `boto3` and `pandas`
- ✅ Built a **scikit-learn pipeline** with `ColumnTransformer` for numeric + categorical columns
- ✅ Engineered **date, text, and interaction features**
- ✅ Split data with **stratified sampling** (no data leakage!)
- ✅ Written **Parquet** outputs back to S3
- ✅ Ingested features into **SageMaker Feature Store** (online + offline)
- ✅ Queried the offline store with **Athena**
- ✅ Trained a **baseline XGBoost** model to validate the feature set

> 🙂 **Who this is for:** You know a little Python and pandas. You've never done feature engineering end-to-end on AWS. That's perfect — we'll go step by step.


## 🔧 Step 0 — Environment Setup

Run the cell below to install any missing packages. Safe to run — pip will skip anything already installed.


In [ ]:
# Install any missing packages.
!pip install -q pandas numpy scikit-learn xgboost boto3 sagemaker pyarrow awswrangler s3fs

### Quick sanity check — import everything and print versions

In [ ]:
import sys, boto3, pandas as pd, numpy as np, sklearn, sagemaker
print("Python      :", sys.version.split()[0])
print("pandas      :", pd.__version__)
print("numpy       :", np.__version__)
print("scikit-learn:", sklearn.__version__)
print("boto3       :", boto3.__version__)
print("sagemaker   :", sagemaker.__version__)

---
## 📦 Module 1 — Set Up S3 Bucket & Upload Dataset

**🆕 v2 fix:** Instead of creating a custom-named bucket (which often fails due to restrictive SageMaker role permissions), we use **SageMaker's default bucket** — this is guaranteed to work with any SageMaker role.

> 📌 **Before running:** make sure `k21_ecommerce_dataset_10000.csv` is uploaded to the **same folder as this notebook** via the Jupyter Upload button.


### Step 1 — Use SageMaker's default bucket (guaranteed write access)

In [ ]:
import boto3
import sagemaker

region = 'us-east-1'

# Use SageMaker's default bucket — your role always has write access to this
sagemaker_session = sagemaker.Session()
bucket = sagemaker_session.default_bucket()

s3 = boto3.client('s3', region_name=region)
print(f'✅ Using bucket: s3://{bucket}')

# Encryption is auto-enabled by AWS on all new buckets since Jan 2023,
# but we try to set it explicitly. If permission is missing, we continue anyway.
try:
    s3.put_bucket_encryption(
        Bucket=bucket,
        ServerSideEncryptionConfiguration={
            'Rules': [{'ApplyServerSideEncryptionByDefault':
                       {'SSEAlgorithm': 'AES256'}}]
        }
    )
    print('✅ Encryption explicitly enabled')
except Exception as e:
    print('ℹ️  Default encryption is auto-enabled by AWS — continuing.')

### Step 2 — Upload the CSV file to S3

In [ ]:
local_file = 'k21_ecommerce_dataset_10000.csv'   # path in current folder
s3_key     = 'k21-lab/raw/k21_ecommerce_dataset_10000.csv'

s3.upload_file(local_file, bucket, s3_key)
print(f'✅ Uploaded to s3://{bucket}/{s3_key}')

# Verify
resp = s3.head_object(Bucket=bucket, Key=s3_key)
print(f"Size: {resp['ContentLength']/1024:.1f} KB")

> ✅ **CHECKPOINT** — You should see the object size printed (~4500 KB).

**Troubleshoot:**
- `FileNotFoundError` → upload the CSV via Jupyter's Upload button first
- `AccessDenied` → your role can't write to the default bucket (very rare) — ask your AWS admin to attach `AmazonSageMakerFullAccess`


---
## 📊 Module 2 — Load & Profile the Data with pandas

Always profile your data before transforming it. Knowing shape, types, and missing values prevents bugs downstream.


### Step 1 — Read directly from S3 into pandas

In [ ]:
import pandas as pd

s3_uri = f's3://{bucket}/{s3_key}'
df = pd.read_csv(s3_uri)
print(f'Shape: {df.shape}')   # Expect (10000, 42)
df.head()

### Step 2 — Quick profile

In [ ]:
df.info()

In [ ]:
print(df.describe().T)

In [ ]:
# Class balance
print(df['churnflag'].value_counts(normalize=True).round(3))

In [ ]:
# Missing-value summary
missing = df.isnull().sum().sort_values(ascending=False)
print(missing[missing > 0])

> 🧐 **Why so many missing values in text columns?** Most customers don't leave complaints or support chats — that's *expected*, not a bug. We'll handle these with sentinel values in Module 3.


### Step 3 — Define column groups

**🆕 v2 fix:** `CAT_HIGH` defined correctly from the start (without `returnreason`) so the target encoder won't crash later.

In [ ]:
# Target
TARGET = 'churnflag'

# Columns to drop (IDs, PII, redundant)
DROP_COLS = [
    'customerid', 'productid', 'orderid',
    'customername', 'email', 'phone',
    'maskedemail', 'maskedphone',
    'productname', 'productdescription',
    'productimageurl', 'audiocallreference',
]

# Numeric features
NUM_COLS = [
    'age', 'loyaltyscore', 'price', 'stock', 'rating',
    'quantity', 'totalamount', 'nextpurchasedays',
    'dailysales', 'monthlyrevenue', 'demandscore',
]

# Low-cardinality categorical → one-hot encode
CAT_LOW = ['gender', 'paymentmethod', 'deliverystatus',
           'customersegment', 'regionaccesstag']

# High-cardinality categorical → target / frequency encode
# NOTE: 'returnreason' is dropped later, so we don't include it here
CAT_HIGH = ['city', 'state', 'category', 'brand']

# Text columns → derive features from these
TEXT_COLS = ['reviewtext', 'complainttext', 'supportchatsummary']

# Date columns
DATE_COLS = ['signupdate', 'orderdate']

# Binary – keep as-is
BIN_COLS = ['suspiciousflag', 'refundrequested', 'piiflag']

print("✅ Column groups defined")

---
## 🧹 Module 3 — Handle Missing Values & Data Types

**🆕 v2 fix:** Added a safety-net pass that fills ANY remaining NaNs, so the final `assert` never trips.


### Step 1 — Convert date columns

In [ ]:
df['signupdate'] = pd.to_datetime(df['signupdate'], errors='coerce')
df['orderdate']  = pd.to_datetime(df['orderdate'],  errors='coerce')
print(df[['signupdate', 'orderdate']].dtypes)

### Step 2 — Impute text columns with sentinel values

In [ ]:
df['reviewtext']         = df['reviewtext'].fillna('No Review')
df['complainttext']      = df['complainttext'].fillna('No Complaint')
df['supportchatsummary'] = df['supportchatsummary'].fillna('None')
df['returnreason']       = df['returnreason'].fillna('Not Applicable')
print("✅ Text columns imputed")

### Step 3 — Impute numeric columns

For `nextpurchasedays`, a missing value means "customer never came back" → large sentinel `999`. Everything else → median imputation (robust to outliers).

In [ ]:
df['nextpurchasedays'] = df['nextpurchasedays'].fillna(999)

for col in NUM_COLS:
    if df[col].isnull().any():
        df[col] = df[col].fillna(df[col].median())

print("✅ Numeric columns imputed")

### Step 4 — Normalize categorical strings + safety-net NaN fill

**🆕 v2 fix:** After normalizing categoricals, we sweep any remaining NaNs in other columns. Columns like `audiocallreference` (which will be dropped in Module 6 anyway) may still have NaNs — this cleanup makes the assertion happy.

In [ ]:
# Strip whitespace and capitalize consistently
for col in CAT_LOW + CAT_HIGH:
    df[col] = df[col].astype(str).str.strip().str.title()

# Safety net: fill any remaining NaNs (mostly in DROP_COLS which we'll remove later)
for col in df.columns:
    if df[col].isnull().any():
        if df[col].dtype == 'object':
            df[col] = df[col].fillna('Unknown')
        elif pd.api.types.is_datetime64_any_dtype(df[col]):
            df[col] = df[col].fillna(pd.Timestamp('1970-01-01'))
        else:
            df[col] = df[col].fillna(0)

# Verify no NaNs remain
assert df.isnull().sum().sum() == 0, 'Still have NaNs!'
print('✅ Clean:', df.shape)

### Step 5 — Cap numeric outliers (1st–99th percentile)

A single customer with a ₹10,000,000 order would skew every numeric feature. We **clip** the top and bottom 1% of values (called *winsorizing*).

> 💡 **Tip:** Always cap **AFTER** imputing. Order matters in feature engineering!

In [ ]:
def cap_percentiles(s, low=0.01, high=0.99):
    q_low, q_high = s.quantile([low, high])
    return s.clip(lower=q_low, upper=q_high)

for col in ['totalamount', 'price', 'monthlyrevenue', 'dailysales']:
    df[col] = cap_percentiles(df[col])

print(df[['totalamount', 'price']].describe().T)

---
## 🔩 Module 4 — Build a scikit-learn Feature Engineering Pipeline

A `Pipeline` chains preprocessing steps into a reusable object. `ColumnTransformer` lets different columns go through different preprocessing steps in parallel.


### Step 1 — Custom Target Encoder for high-cardinality categoricals

Replaces each category (e.g., city name) with the average churn rate for that category. **Smoothing** protects rare categories from overfitting.

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin

class TargetEncoder(BaseEstimator, TransformerMixin):
    """Mean target encoder with additive smoothing to avoid overfitting rare categories."""
    def __init__(self, cols, smoothing=10):
        self.cols, self.smoothing = cols, smoothing
        self.maps_, self.global_mean_ = {}, None

    def fit(self, X, y):
        self.global_mean_ = float(y.mean())
        for c in self.cols:
            stats = pd.DataFrame({c: X[c], 't': y}).groupby(c)['t'].agg(['mean', 'count'])
            smoothed = (stats['count'] * stats['mean'] +
                        self.smoothing * self.global_mean_) / (stats['count'] + self.smoothing)
            self.maps_[c] = smoothed.to_dict()
        return self

    def transform(self, X):
        Xc = X.copy()
        for c in self.cols:
            Xc[c] = Xc[c].map(self.maps_[c]).fillna(self.global_mean_)
        return Xc

print("✅ TargetEncoder class defined")

### Step 2 — Bucketize age into bands

In [ ]:
def add_age_bucket(df):
    df = df.copy()
    df['age_bucket'] = pd.cut(
        df['age'],
        bins=[0, 25, 40, 60, 200],
        labels=['18-25', '26-40', '41-60', '60+']
    ).astype(str)
    return df

df = add_age_bucket(df)
print(df['age_bucket'].value_counts())

### Step 3 — Date feature extraction

Raw dates are useless to a model. Their components (year, month, weekday, is-weekend) are highly predictive.

In [ ]:
def extract_date_features(df):
    df = df.copy()
    for col in DATE_COLS:
        df[f'{col}_year']       = df[col].dt.year
        df[f'{col}_month']      = df[col].dt.month
        df[f'{col}_day']        = df[col].dt.day
        df[f'{col}_dayofweek']  = df[col].dt.dayofweek
        df[f'{col}_is_weekend'] = (df[col].dt.dayofweek >= 5).astype(int)

    df['customer_tenure_days'] = (df['orderdate'] - df['signupdate']).dt.days
    return df

df = extract_date_features(df)
print(df.filter(like='signupdate_').head())

---
## 🧠 Module 5 — Derived / Interaction / Text-based Features

This is where domain knowledge pays off. Single raw columns are rarely the most predictive — combining them or deriving flags gives the model stronger signals.


### Step 1 — Business-logic derived features

In [ ]:
def add_derived_features(df):
    df = df.copy()

    # Price / quantity relationships
    df['avg_price_per_item']  = df['totalamount'] / df['quantity'].clip(lower=1)
    df['discount_proxy']      = (df['price'] - df['avg_price_per_item']).clip(lower=0)

    # Engagement flags from text
    df['has_review']          = (df['reviewtext']    != 'No Review').astype(int)
    df['has_complaint']       = (df['complainttext'] != 'No Complaint').astype(int)
    df['has_support_chat']    = (df['supportchatsummary'] != 'None').astype(int)

    # Text lengths
    df['review_length']       = df['reviewtext'].str.len()
    df['complaint_length']    = df['complainttext'].str.len()

    # Composite risk score
    df['risk_score']          = df['suspiciousflag'] + df['refundrequested'] + df['has_complaint']

    # Binary flags
    df['high_value_customer'] = (df['loyaltyscore'] >= 70).astype(int)
    df['low_rating_order']    = (df['rating']       <= 2.5).astype(int)

    # Interaction
    df['price_x_rating']      = df['price'] * df['rating']

    return df

df = add_derived_features(df)
print('✅ Derived features created')

### Step 2 — Frequency encoding

In [ ]:
for col in ['city', 'state', 'brand', 'category']:
    freq_map = df[col].value_counts(normalize=True).to_dict()
    df[f'{col}_freq'] = df[col].map(freq_map)

print("✅ Frequency-encoded columns added:",
      [c for c in df.columns if c.endswith('_freq')])

### Step 3 — Simple sentiment signal from reviews

Dependency-free keyword counts. In production, swap for a pre-trained model (e.g., DistilBERT from SageMaker JumpStart).

In [ ]:
positive_words = {'excellent','great','love','amazing','fast','perfect','good','awesome','happy','superb'}
negative_words = {'bad','poor','slow','damaged','late','worst','awful','broken','defective','terrible'}

def count_words(text, vocab):
    if not isinstance(text, str):
        return 0
    return sum(1 for w in text.lower().split() if w.strip('.,!?') in vocab)

df['review_pos_words'] = df['reviewtext'].apply(lambda t: count_words(t, positive_words))
df['review_neg_words'] = df['reviewtext'].apply(lambda t: count_words(t, negative_words))
df['review_sentiment'] = df['review_pos_words'] - df['review_neg_words']
print(df[['review_sentiment']].describe().T)

---
## 🪓 Module 6 — Train/Validation/Test Split & Save to S3

**🆕 v2 fix:** Target encoder uses correct `CAT_HIGH` (without `returnreason`).

**Why split BEFORE encoding?** Target encoding uses `churnflag` values. If fit on the full dataset, target info leaks into val/test features, inflating scores.


### Step 1 — Stratified split

In [ ]:
from sklearn.model_selection import train_test_split

# Drop columns we shouldn't feed to the model
ENGINEERED_DROP = DROP_COLS + TEXT_COLS + DATE_COLS + ['returnreason']
X = df.drop(columns=ENGINEERED_DROP + [TARGET])
y = df[TARGET]

# 70% / 15% / 15% with stratification on target
X_train, X_tmp, y_train, y_tmp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=42)
X_val, X_test, y_val, y_test   = train_test_split(
    X_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=42)

print(f'Train: {X_train.shape}   Val: {X_val.shape}   Test: {X_test.shape}')
print('Class balance preserved:',
      round(y_train.mean(),3), round(y_val.mean(),3), round(y_test.mean(),3))

### Step 2 — Fit target encoder on TRAIN ONLY

> ⚠️ Fitting on the full dataset leaks target info. Always fit encoders on training data only.

In [ ]:
te = TargetEncoder(cols=CAT_HIGH, smoothing=10)
te.fit(X_train, y_train)

X_train = te.transform(X_train)
X_val   = te.transform(X_val)
X_test  = te.transform(X_test)

print("✅ Target encoder fitted on train, applied to val & test")
print("Encoded columns:", CAT_HIGH)

### Step 3 — Fit the full scikit-learn pipeline

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# age_bucket is low-card categorical → add to CAT_LOW
CAT_LOW_FULL = CAT_LOW + ['age_bucket']

preproc = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(),                           NUM_COLS +
                                                            ['customer_tenure_days',
                                                             'avg_price_per_item',
                                                             'discount_proxy',
                                                             'review_length',
                                                             'complaint_length',
                                                             'price_x_rating',
                                                             'review_sentiment']),
        ('cat', OneHotEncoder(handle_unknown='ignore',
                              sparse_output=False),         CAT_LOW_FULL),
        ('bin', 'passthrough',                              BIN_COLS + ['has_review',
                                                                        'has_complaint',
                                                                        'has_support_chat',
                                                                        'high_value_customer',
                                                                        'low_rating_order']),
        ('hi',  'passthrough',                              CAT_HIGH),
    ],
    remainder='drop',
    verbose_feature_names_out=False,
)

preproc.fit(X_train)

X_train_t = pd.DataFrame(preproc.transform(X_train),
                         columns=preproc.get_feature_names_out())
X_val_t   = pd.DataFrame(preproc.transform(X_val),
                         columns=preproc.get_feature_names_out())
X_test_t  = pd.DataFrame(preproc.transform(X_test),
                         columns=preproc.get_feature_names_out())

print(f'✅ Engineered feature count: {X_train_t.shape[1]}')

### Step 4 — Write Parquet files back to S3

**Parquet** is columnar — ~10× smaller and ~5× faster than CSV. The ML industry standard.

In [ ]:
for name, Xp, yp in [
    ('train', X_train_t, y_train),
    ('val',   X_val_t,   y_val),
    ('test',  X_test_t,  y_test)
]:
    out = Xp.copy()
    out[TARGET] = yp.values
    s3_out = f's3://{bucket}/k21-lab/features/{name}/{name}.parquet'
    out.to_parquet(s3_out, index=False)
    print(f'  {name}: {out.shape} → {s3_out}')

> ✅ **CHECKPOINT** — Three Parquet files now sit under `s3://<bucket>/k21-lab/features/`.

---
## 🗄️ Module 7 — Register Features in SageMaker Feature Store

**🆕 v2 fix:** Reconnect-by-name pattern. If the feature group already exists (e.g., from a previous run), we reconnect instead of crashing. Progress messages so the cell never looks frozen.

> 💡 This module requires `AmazonSageMakerFeatureStoreAccess` IAM policy. If you don't have it, skip to Module 9 — XGBoost works without Feature Store.


### Step 1 — Build a customer-level feature frame

In [ ]:
cust = df.groupby('customerid').agg(
    avg_totalamount   = ('totalamount',        'mean'),
    sum_quantity      = ('quantity',           'sum'),
    loyalty           = ('loyaltyscore',       'max'),
    avg_tenure        = ('customer_tenure_days','mean'),
    has_complaint_any = ('has_complaint',      'max'),
    risk_score_max    = ('risk_score',         'max'),
    review_sent_mean  = ('review_sentiment',   'mean'),
    order_count       = ('churnflag',          'count'),
    churn_last        = ('churnflag',          'last'),
).reset_index()

import time
cust['event_time'] = float(time.time())
print(f'✅ Customer-level frame: {cust.shape}')

### Step 2 — Create OR reconnect to feature group (idempotent)

This cell handles three scenarios:
1. Feature group doesn't exist → create it
2. Feature group exists and is Created → reconnect (no-op)
3. Feature group exists and is Creating → wait for it

In [ ]:
from sagemaker.feature_store.feature_group import FeatureGroup
from sagemaker.session import Session
import sagemaker, time, sys

sagemaker_session = Session()
role = sagemaker.get_execution_role()

fg_name = 'k21-customer-features'
fg = FeatureGroup(name=fg_name, sagemaker_session=sagemaker_session)

# Prepare dtypes for Feature Store
for c in cust.select_dtypes(include='object').columns:
    cust[c] = cust[c].astype('string')

# Check if feature group already exists
try:
    existing_status = fg.describe().get('FeatureGroupStatus')
    print(f'📋 Feature group already exists — status: {existing_status}')
    need_create = False
except Exception as e:
    if 'ResourceNotFound' in str(e):
        print('📋 Feature group does not exist — will create')
        need_create = True
    else:
        raise

# Create if needed
if need_create:
    fg.load_feature_definitions(data_frame=cust)
    print('🚀 Creating feature group...')
    fg.create(
        s3_uri                  = f's3://{bucket}/k21-lab/features/offline/',
        record_identifier_name  = 'customerid',
        event_time_feature_name = 'event_time',
        role_arn                = role,
        enable_online_store     = True,
        description             = 'K21 e-commerce customer-level features',
    )

# Wait for 'Created' status
elapsed = 0
while fg.describe().get('FeatureGroupStatus') == 'Creating':
    time.sleep(5)
    elapsed += 5
    print(f'   ⏳ Still creating... ({elapsed}s elapsed)')
    sys.stdout.flush()

final_status = fg.describe()['FeatureGroupStatus']
print(f'\n✅ Feature group status: {final_status}')
if final_status == 'CreateFailed':
    print('❌ Failure reason:', fg.describe().get('FailureReason', 'Unknown'))

### Step 3 — Ingest the data

In [ ]:
fg.ingest(data_frame=cust, max_workers=3, wait=True)
print(f'✅ Ingested {len(cust)} records into {fg_name}')

### Step 4 — Online lookup (real-time)

In [ ]:
runtime = boto3.client('sagemaker-featurestore-runtime')
sample_id = cust['customerid'].iloc[0]

resp = runtime.get_record(
    FeatureGroupName=fg_name,
    RecordIdentifierValueAsString=str(sample_id),
)
print(f'✅ Online lookup for customer {sample_id}:')
print('-' * 50)
for r in resp['Record']:
    print(f"  {r['FeatureName']:25s}: {r['ValueAsString']}")

> ✅ **CHECKPOINT** — Customer attributes returned in milliseconds = online store is serving correctly.

---
## 🔍 Module 8 — Query the Feature Store with Athena

The offline store auto-registers a Glue table — queryable with SQL via Athena.

> ⏰ **Heads-up:** Feature Store has a **~15-minute ingestion delay** before data appears in the offline store. If the query returns zero rows, wait 15 min and re-run.


### Step 1 — Get Athena connection details

In [ ]:
desc = fg.describe()
offline_cfg = desc['OfflineStoreConfig']['DataCatalogConfig']
print('Database :', offline_cfg['Database'])
print('Table    :', offline_cfg['TableName'])
print('Catalog  :', offline_cfg['Catalog'])

### Step 2 — Run a query

In [ ]:
import awswrangler as wr

sql = f"""
    SELECT customerid,
           avg_totalamount,
           loyalty,
           has_complaint_any,
           risk_score_max,
           churn_last
    FROM   "{offline_cfg['Database']}"."{offline_cfg['TableName']}"
    ORDER  BY avg_totalamount DESC
    LIMIT  10
"""

result = wr.athena.read_sql_query(
    sql=sql,
    database=offline_cfg['Database'],
    s3_output=f's3://{bucket}/k21-lab/athena-results/',
)

if len(result) == 0:
    print('⏳ Offline store not yet populated. Wait ~15 minutes and re-run this cell.')
else:
    print(result)

---
## 🧪 Module 9 — Sanity-Check with a Baseline XGBoost Model

If a quick baseline gives random accuracy (~50%), your features have a problem. Don't spend time tuning before the baseline works.


### Step 1 — Train XGBoost classifier

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, classification_report
)

model = XGBClassifier(
    n_estimators   = 400,
    max_depth      = 6,
    learning_rate  = 0.1,
    eval_metric    = 'logloss',
    random_state   = 42,
    n_jobs         = -1,
)

model.fit(X_train_t, y_train,
          eval_set=[(X_val_t, y_val)], verbose=False)

preds = model.predict(X_test_t)
proba = model.predict_proba(X_test_t)[:, 1]

print(f'Accuracy : {accuracy_score(y_test, preds):.4f}')
print(f'F1       : {f1_score(y_test, preds):.4f}')
print(f'ROC-AUC  : {roc_auc_score(y_test, proba):.4f}')
print(classification_report(y_test, preds))

### Step 2 — Plot top-20 feature importances

Gold mine chart. Check that your engineered features (`avg_price_per_item`, `risk_score`, `customer_tenure_days`) rank highly — if they do, your feature engineering was worth the effort.

In [ ]:
import matplotlib.pyplot as plt

fi = pd.Series(model.feature_importances_, index=X_train_t.columns)
top = fi.sort_values(ascending=False).head(20)

plt.figure(figsize=(8, 6))
top.sort_values().plot(kind='barh')
plt.title('Top 20 Feature Importances — XGBoost baseline')
plt.xlabel('Gain')
plt.tight_layout()
plt.show()

> ✅ **CHECKPOINT** — Expected baseline: **F1 ≈ 0.82**, **ROC-AUC ≈ 0.88**.


---
## 💾 Save & Reuse Your Pipeline

Save the fitted pipeline and target encoder with `joblib`. At inference time, load the exact same preprocessing — so training and serving stay consistent.

In [ ]:
import joblib

joblib.dump(preproc, 'k21_preprocessing_pipeline.joblib')
joblib.dump(te,      'k21_target_encoder.joblib')

s3.upload_file('k21_preprocessing_pipeline.joblib', bucket, 'k21-lab/artifacts/preproc.joblib')
s3.upload_file('k21_target_encoder.joblib',         bucket, 'k21-lab/artifacts/target_encoder.joblib')
print('✅ Artifacts saved to S3')

---
## 🧽 Clean-Up

**🆕 v2 fix:** Robust cleanup that survives kernel restarts. Finds and deletes ALL lab resources automatically.

> ⚠️ SageMaker resources keep billing until stopped. Run these cells when you're done.


### 🧹 One-cell cleanup — deletes feature group + ALL k21 buckets

In [ ]:
import boto3

s3 = boto3.client('s3')
s3_res = boto3.resource('s3')

# --- Delete feature group ---
try:
    from sagemaker.feature_store.feature_group import FeatureGroup
    from sagemaker.session import Session
    fg = FeatureGroup(name='k21-customer-features', sagemaker_session=Session())
    fg.delete()
    print('✅ Deleted feature group: k21-customer-features')
except Exception as e:
    if 'ResourceNotFound' in str(e):
        print('✅ Feature group already deleted')
    else:
        print(f'⚠️  Feature group: {e}')

# --- Delete all k21 buckets (safe pattern match) ---
all_buckets = [b['Name'] for b in s3.list_buckets()['Buckets']]
lab_buckets = [b for b in all_buckets if 'k21-featureeng' in b]

if lab_buckets:
    print(f'\nFound {len(lab_buckets)} k21-featureeng bucket(s):')
    for b in lab_buckets:
        try:
            bucket_obj = s3_res.Bucket(b)
            bucket_obj.objects.all().delete()
            bucket_obj.object_versions.all().delete()
            bucket_obj.delete()
            print(f'  ✅ Deleted: {b}')
        except Exception as e:
            print(f'  ⚠️  {b}: {e}')
else:
    print('\n✅ No k21-featureeng buckets to delete')

# --- Clean up k21-lab/ prefix from SageMaker default bucket ---
try:
    default_bucket = sagemaker.Session().default_bucket()
    bucket_obj = s3_res.Bucket(default_bucket)
    deleted_count = 0
    for obj in bucket_obj.objects.filter(Prefix='k21-lab/'):
        obj.delete()
        deleted_count += 1
    if deleted_count:
        print(f'\n✅ Deleted {deleted_count} objects from s3://{default_bucket}/k21-lab/')
except Exception as e:
    print(f'\n⚠️  Default bucket cleanup: {e}')

print('\n🎉 Cleanup complete!')

### 🔴 Final step — Stop the Notebook Instance (MANUAL)

Python can't stop the notebook instance it's running on. Do this in the AWS Console:

1. AWS Console → **SageMaker**
2. Left sidebar → **Notebook → Notebook instances**
3. Select `k21-featureeng-notebook`
4. **Actions → Stop** (or **Delete** if you're fully done)

**This is the only resource that costs real money** if left running ($36/month for `ml.t3.medium`).

### 💰 Verify spend

Visit [console.aws.amazon.com/billing/home#/budgets](https://console.aws.amazon.com/billing/home#/budgets) → current month should be **< $5**.


---
## 🎉 Summary — What You Built

- ✅ A reproducible Python pipeline loading raw K21 data from S3 and emitting ML-ready Parquet
- ✅ **20+ engineered features** across numeric, categorical, text, date, and interaction dimensions
- ✅ A **leak-free** train/val/test split with stratified sampling
- ✅ A **SageMaker Feature Group** serving customer features both online and offline
- ✅ A validated **XGBoost baseline** proving the feature set is predictive

### Next steps

- **Train on SageMaker** — package this code into a SageMaker Training Job
- **Hyperparameter tuning** — use SageMaker Automatic Model Tuning
- **SageMaker Pipelines** — wrap each module as a pipeline step for full MLOps
- **Real-time inference** — deploy the model as a SageMaker endpoint
- **Monitoring** — use SageMaker Model Monitor for feature-drift detection

> ✅ **You completed the K21 Academy Feature Engineering lab!** Push this notebook to GitHub and tag **@K21Academy** on LinkedIn. 🚀

---
*© K21 Academy — Job-Oriented AI/ML Programs — [k21academy.com](https://k21academy.com)*
